<a href="https://colab.research.google.com/github/besturkmen/AcademiQ_Data_Science/blob/main/AcademiQ_Data_Science_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re # metinlerde desen arama ve temizleme
import time # kod çalışma süresini hesaplar
import warnings # uyarı mesajlarını gizlemek ve yönetmek için
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.compose import ColumnTransformer # sayısal ve kategorik sütunlara farklı işlemler uygulamak için
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # eksik verileri doldurmak için
from sklearn.preprocessing import OrdinalEncoder # Kategorik verileri sayısala çevirmek için
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
df =pd.read_csv("Food_Delivery_Times.csv")
df.head()

,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84
2,741,9.52,Foggy,Low,Night,Scooter,28,1.0,59
3,661,7.44,Rainy,Medium,Afternoon,Scooter,5,1.0,37
4,412,19.03,Clear,Low,Morning,Bike,16,5.0,68


In [3]:
def clean_col(c):
  c = c.lower().strip() #lower tüm harfleri küçültecek , strip ise baştaki ve sondaki boşlukları silecek
  c = re.sub(r"[^a-z0-9]+" , "_" , c)
  return c.strip("_")
df.columns = [clean_col(c) for c in df.columns]

print(df.columns.tolist())



['order_id', 'distance_km', 'weather', 'traffic_level', 'time_of_day', 'vehicle_type', 'preparation_time_min', 'courier_experience_yrs', 'delivery_time_min']


In [53]:
#target_candidates = [c for c in df.columns if "time" in c]

#print(target_candidates)

#TARGET= target_candidates[0]



In [54]:
#df[TARGET]= (
   # df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)
#)

#df = df.dropna(subset=[TARGET]) #NaN olan ifadeleri veri setinden kaldırır.
#print(df[TARGET].head())
# 30 mins -> 30
# 12.569 minutes -> 12.569
# 450 min -> 450

In [4]:
TARGET = "delivery_time_min"
print("Target:", TARGET)

Target: delivery_time_min


In [5]:
df[TARGET]= (
    df[TARGET].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)
)

In [6]:
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print("Target ilk değerler:")
display(df[TARGET].head())


Target ilk değerler:


,delivery_time_min
0,43.0
1,84.0
2,59.0
3,37.0
4,68.0


In [9]:
drop_like_id = [c for c in df.columns if c in ("id" , "person_id") or c.endswith("_id")]
df = df.drop(columns=drop_like_id, errors="ignore")

print("ID sonrası şekil:" , df.shape)

ID sonrası şekil: (1000, 8)
